In [1]:
import sympy as sp
import anaflow
import numpy as np
import matplotlib.pyplot as plt
import scipy.special as sc

In [3]:
# определим переменные с которыми будем работать 


r_d = sp.symbols('r_D', real=True, positive=True)
t_d = sp.symbols('t_D', real=True, positive=True)
p_d = sp.symbols('p_D', cls=sp.Function, real=True, positive=True)
Lp_d = sp.symbols(r"{\widetilde{p_{D}}}", cls=sp.Function, real=True, positive=True)
q_d = sp.symbols('q_D', real=True, positive=True)
Lq_d = sp.symbols(r"{\widetilde{q_{D}}}", real=True, positive=True)

r_w, p_i, q_ref = sp.symbols('r_w p_i q_ref', real=True, positive=True)

r_wd = sp.symbols('r_wd', real=True, positive=True)
pi = sp.symbols('pi', real=True, positive=True)
u = sp.symbols('u',real=True, positive=True)
a, b = sp.symbols('a b', real=True, positive=True)


r = sp.symbols('r', real=True, positive=True)
t = sp.symbols('t', real=True, positive=True)
k, phi, mu, c_t, h = sp.symbols('k phi mu c_t h', real=True, positive=True)
p = sp.symbols('p', cls=sp.Function, real=True, positive=True)


a_d = sp.symbols('a_D', cls=sp.Function, real=True, positive=True)

pi = sp.pi

In [4]:
# определим уравнение фильтрации в пространстве Лапласа
diff_eq = sp.Eq(u * Lp_d(r_d),
                1 / r_d * (sp.diff(r_d * sp.diff(Lp_d(r_d), r_d) , r_d)) )
diff_eq

Eq(u*{\widetilde{p_{D}}}(r_D), (r_D*Derivative({\widetilde{p_{D}}}(r_D), (r_D, 2)) + Derivative({\widetilde{p_{D}}}(r_D), r_D))/r_D)

In [5]:
# попробуем решить уравнение 
soln = sp.dsolve(diff_eq, Lp_d(r_d))
soln

Eq({\widetilde{p_{D}}}(r_D), C1*besseli(0, r_D*sqrt(u)) + C2*bessely(0, I*r_D*sqrt(u)))

In [6]:
# зададим в явном виде решение с использованием K_0 и I_0
A, B = sp.symbols('A B')
soln2 = sp.Eq(Lp_d(r_d) , A * sp.besselk(0, r_d * sp.sqrt(u)) + B * sp.besseli(0, r_d * sp.sqrt(u)))
soln2

Eq({\widetilde{p_{D}}}(r_D), A*besselk(0, r_D*sqrt(u)) + B*besseli(0, r_D*sqrt(u)))

In [7]:
soln3 = soln2.subs(B, 0)
print('Общее решение с учетом условия на бесконечности')
display(soln3)

Общее решение с учетом условия на бесконечности


Eq({\widetilde{p_{D}}}(r_D), A*besselk(0, r_D*sqrt(u)))

In [10]:
a_d

a_D

In [13]:
print('граничное условие на скважине')
bc_well = sp.Eq( 2* pi *r_d * sp.diff(p_d(t_d, r_d), r_d) ,-a_d(t_d))
display(bc_well)
print('применим преобразование Лапласа к обеим частям граничного условия')
eq_boundary_Laplace = sp.Eq(sp.laplace_transform(bc_well.lhs, t_d, u,  noconds=True) ,  
                            sp.laplace_transform(bc_well.rhs, t_d, u,  noconds=True))
display(eq_boundary_Laplace)
print('Выражение для граничного условия на скважине')
bc2 = sp.Eq( 2* pi *r_d * soln3.rhs.diff(r_d), eq_boundary_Laplace.rhs)
display(bc2)
print('найдем A')
bc2_sol = sp.solve(bc2.subs(r_d, 1), A)
display(sp.Eq(A,bc2_sol[0]))
print('частное решение для любого расстояния')
soln4 = soln3.subs(A, bc2_sol[0])
display(soln4)
print('частное решение для забойного давления')
soln5 = soln4.subs(r_d, 1)
display(soln5)


print('выражение для дебита')
sol_qd = sp.Eq(Lq_d, (-2 * pi * r_d * sp.diff( soln4.rhs, r_d)).subs(r_d,1))
display(sol_qd)

граничное условие на скважине


Eq(2*pi*r_D*Derivative(p_D(t_D, r_D), r_D), -a_D(t_D))

применим преобразование Лапласа к обеим частям граничного условия


Eq(2*pi*r_D*LaplaceTransform(Derivative(p_D(t_D, r_D), r_D), t_D, u), -LaplaceTransform(a_D(t_D), t_D, u))

Выражение для граничного условия на скважине


...

Eq({\widetilde{p_{D}}}(r_D), LaplaceTransform(a_D(t_D), t_D, u)*besselk(0, r_D*sqrt(u))/(2*pi*sqrt(u)*besselk(1, sqrt(u))))

частное решение для забойного давления


Eq({\widetilde{p_{D}}}(1), LaplaceTransform(a_D(t_D), t_D, u)*besselk(0, sqrt(u))/(2*pi*sqrt(u)*besselk(1, sqrt(u))))

выражение для дебита


Eq({\widetilde{q_{D}}}, LaplaceTransform(a_D(t_D), t_D, u))

In [14]:
print('граничное условие на скважине')
bc_well = sp.Eq( p_d(t_d, r_d) , a_d(t_d))
display(bc_well)
print('применим преобразование Лапласа к обеим частям граничного условия')
eq_boundary_Laplace = sp.Eq(sp.laplace_transform(bc_well.lhs, t_d, u,  noconds=True) ,  
                            sp.laplace_transform(bc_well.rhs, t_d, u,  noconds=True))
display(eq_boundary_Laplace)
print('Выражение для граничного условия на скважине')
bc2 = sp.Eq( soln3.rhs, eq_boundary_Laplace.rhs)
display(bc2)
print('найдем A')
bc2_sol = sp.solve(bc2.subs(r_d, r_wd), A)
display(sp.Eq(A,bc2_sol[0]))
print('частное решение для любого расстояния и условия на забое r_wd=1')
soln4 = soln3.subs(A, bc2_sol[0]).subs(r_wd, 1)
display(soln4)
print('частное решение для забойного давления')
soln5 = soln4.subs(r_wd, 1).subs(r_d, 1)
display(soln5)

print('выражение для дебита')
sol_qd = sp.Eq(Lq_d, (-2 * pi * r_d * sp.diff( soln4.rhs, r_d)).subs(r_d,1))
display(sol_qd)

граничное условие на скважине


Eq(p_D(t_D, r_D), a_D(t_D))

применим преобразование Лапласа к обеим частям граничного условия


Eq(LaplaceTransform(p_D(t_D, r_D), t_D, u), LaplaceTransform(a_D(t_D), t_D, u))

Выражение для граничного условия на скважине


...

Eq({\widetilde{p_{D}}}(r_D), LaplaceTransform(a_D(t_D), t_D, u)*besselk(0, r_D*sqrt(u))/besselk(0, sqrt(u)))

частное решение для забойного давления


Eq({\widetilde{p_{D}}}(1), LaplaceTransform(a_D(t_D), t_D, u))

выражение для дебита


Eq({\widetilde{q_{D}}}, 2*pi*sqrt(u)*LaplaceTransform(a_D(t_D), t_D, u)*besselk(1, sqrt(u))/besselk(0, sqrt(u)))